In [0]:
orders = spark.read.table("workspace.ecompulse_bronze.orders")

In [0]:
import pyspark.sql.functions as F

In [0]:
orders = orders.withColumn(
    "order_status",
    F.when(F.lower(F.col("order_status")) == "canceled", "cancelled")
     .otherwise(F.lower(F.col("order_status")))
)

In [0]:
drop_columns = ["order_approved_at", "order_delivered_carrier_date"]

orders = orders.drop(*drop_columns)

In [0]:
orders = orders.withColumn(
    "order_delivered_customer_date",
    F.when(
        F.col("order_status") == "delivered", 
        F.coalesce(F.col("order_delivered_customer_date"), F.col("order_estimated_delivery_date"))
    ).otherwise(F.col("order_delivered_customer_date"))
)

In [0]:
orders = orders.withColumns({
    "order_purchase_date": F.to_date(F.col("order_purchase_timestamp")),
    "order_purchase_hour": F.hour(F.col("order_purchase_timestamp")),
    "order_delivered_date": F.to_date(F.col("order_delivered_customer_date")),
    "order_delivered_hour": F.hour(F.col("order_delivered_customer_date"))
})

In [0]:
orders.write.mode("overwrite").saveAsTable("workspace.ecompulse_silver.orders")

In [0]:
%sql
select * from workspace.ecompulse_silver.orders limit 5;